## Preprocess data

This notebook expects to take standard input data from the corridor model and analytics pipelines and output tables that are ready for the front end to use. Most notably this includes creating regional clusters and tagging each asset to them

### Import necessary functions

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

### Little hack to reproducibly set working dir as the project directory (one level up) in notebooks
if "original_dir" not in vars():
    original_dir = os.path.abspath("")

os.chdir(original_dir)
os.chdir("..")  ### Adjust as needed to get to root
print(f"Project root (ensure this is correct): {os.getcwd()}")

import src.preprocessing.data_preprocessing as funcs


import geopandas as gpd
import pandas as pd
import numpy as np
import random
import sqlite3
from shapely import wkt
import re

from src.general_utilities import io_utils, logging_utils

random.seed(42)

pd.set_option("display.max_columns", 999)

#### Make DB Connection

In [ ]:
conn = sqlite3.connect("data/F0_raw/digital_twin_database.sqlite")

#### Areas of Jurisdiction

In [ ]:
aoj = io_utils.load_file_from_catalog("areas_of_jurisdiction")
aoj["NAME"] = aoj["NAME"].map(lambda x: re.sub(r"\.", "", x)) # periods in names can cause issues
aoj.head(3)

In [ ]:
aoj[["NAME"]].to_sql("areas_of_jurisdiction", conn, if_exists="replace", index=False)

## Create the output table for each solution

We'll want tables that the app needs stored in the db and tables that our pipeline will need stored as flat files via the catalog

### Veg Management

Veg comes with a few files:
* The corridor definitions
* The units of work aggregation
* The optimized trimming cycles to aim for

We need all of these in the database.

**Temporary solution**
We need to revisit the vegX pipeline itself to output a nice clean file format by default so this data wrangling here is not needed.

In [ ]:
corridors_df = io_utils.load_file_from_catalog("front_end_input")
print(corridors_df.shape)
corridors_df.head(3)

In [ ]:
corridors_df = corridors_df.rename(columns={"line_geometries": "geometry"})  # temp fix
corridors_df = gpd.GeoDataFrame(corridors_df, geometry="geometry")
corridors_df.head()

In [ ]:
veg_df = funcs.create_features_for_veg(corridors_df)
veg_df["average vegetation density (%) [MJM]"] = (
    veg_df["average vegetation density (%) [MJM]"] / 100
)
veg_df.head()

### Create bins 

In [ ]:
columns_to_categorize = [
    "probability of outage (%)",  # probability of failure
    "risk (customer interruptions)",  # risk
    "customers affected (adjusted)",  # criticality
]

for column in columns_to_categorize:
    veg_df = funcs.categorize_by_deciles(df=veg_df, column=column)
veg_df.head()

#### Add in scenario outcomes

In [ ]:
import re

scenario_outcomes = io_utils.load_file_from_catalog("optimization_scenarios")
scenario_outcomes = scenario_outcomes[
    ["nearest_upstream_device", "scenario", "frequency"]
].rename(columns={"nearest_upstream_device": "nearest upstream device"})
scenario_outcomes = (
    scenario_outcomes.groupby(["nearest upstream device", "scenario"])["frequency"]
    .agg("first")
    .unstack()
)
scenario_outcomes.columns = [
    re.sub("_", " ", x) + " frequency" for x in scenario_outcomes.columns
]
scenario_outcomes = scenario_outcomes.reset_index()
scenario_outcomes.head(3)

In [ ]:
veg_df = veg_df.merge(scenario_outcomes, how="left", on="nearest upstream device")
veg_df.head(3)

In [ ]:
# map the names to frequncies

# Mapping dictionary
frequency_mapping = {
    "T1": "three times a year",
    "T2": "two times a year",
    "T3": "one time a years",
    "T4": "one time a years",
    "T5": "one time a years",
}

# Apply mapping to each frequency column
for col in ["hybrid frequency", "scenario 1 frequency", "scenario 2 frequency"]:
    veg_df[col] = veg_df[col].map(frequency_mapping)

veg_df.head()

#### Export to file

In [ ]:
veg_out = veg_df.drop(
    [
        "line type",
        "number lines",
        "count of outages last year",
        "count of outages last two years",
        "count of outages last three years",
        "criticality",
    ],
    axis=1,
)

### organize corridor view into organized columns 

In [ ]:
veg_out = veg_out[
    [
        "feeder id",
        "nearest upstream device",
        "geometry",
        "corridor length",
        "average vegetation density (%) [META + Sentinel-2]",
        "average vegetation density (%) [MJM]",
        "probability of outage (%)",
        "probability of outage (%) [bins]",
        "cost to trim (BHT) [META + Sentinel-2]",
        "cost to trim (BHT) [MJM]",
        "risk (customer interruptions)",
        "risk (customer interruptions) [bins]",
        "customers affected (original)",
        "customers affected (adjusted)",
        "customers affected (adjusted) [bins]",
        "scenario 1 frequency",
        "scenario 2 frequency",
        "hybrid frequency",
    ]
].rename(
    columns={
        "average vegetation density (%) [META + Sentinel-2]": "vegetation density (%) [META + Sentinel-2]",
        "average vegetation density (%) [MJM]": "vegetation density (%) [MJM]",
        "scenario 1 frequency": "cost focus scenario frequency",
        "scenario 2 frequency": "reliability focus scenario frequency",
        "hybrid frequency": "hybrid scenario frequency",
    }
)

In [ ]:
io_utils.save_file_via_catalog(veg_out, "veg_features")

#### Export corridor file to sqllite database

Fill in Nans, cast to basic types and stringify the geometry

It's basically the same table as the above though. I just keep the parquet around for convenience on the map generation pipeline. The app will use the DB.

In [ ]:
# Select non-categorical columns
non_categorical_cols = veg_out.select_dtypes(exclude=["category"]).columns

# Fill NaN for non-categorical columns only
veg_out[non_categorical_cols] = veg_out[non_categorical_cols].fillna(0)

# Convert to the desired CRS
veg_corridors_to_sql = veg_out.to_crs("epsg:4326")

veg_corridors_to_sql["feeder id"] = veg_corridors_to_sql["feeder id"].apply(
    lambda x: str(x)
)
veg_corridors_to_sql["geometry_wkt"] = veg_corridors_to_sql["geometry"].map(
    lambda x: x.wkt
)
veg_corridors_to_sql = veg_corridors_to_sql.drop("geometry", axis=1)


veg_corridors_to_sql.map(lambda x: round(x, 2) if type(x) == float else x).to_sql(
    "veg_features", conn, if_exists="replace", index=False
)
veg_corridors_to_sql.head()

#### Make a feeder level file

Here we do a groupby agg to get feeder level features for the first maps

#### Create length weighted score so we can give a sensible estimate of "mean" density for a feeder

In [ ]:
veg_df["share line miles"] = veg_df["corridor length"] / veg_df.groupby("feeder id")[
    "corridor length"
].transform("sum")

veg_df["weighted density (%) [META + Sentinel-2]"] = (
    veg_df["average vegetation density (%) [META + Sentinel-2]"]
    * veg_df["share line miles"]
)

veg_df["weighted density (%) [MJM]"] = (
    veg_df["average vegetation density (%) [MJM]"] * veg_df["share line miles"]
)

veg_df["weighted probability"] = (
    veg_df["probability of outage (%)"] * veg_df["share line miles"]
)
veg_df.head()

#### Create feeder level features

In [ ]:
veg_units = veg_df[["feeder id", "geometry"]].dissolve("feeder id")

veg_units = veg_units.merge(
    veg_df.groupby("feeder id").agg(
        risk=pd.NamedAgg("risk (customer interruptions)", "sum"),
        average_probability_of_outage=pd.NamedAgg("weighted probability", "sum"),
        total_original_customer_count=pd.NamedAgg(
            "customers affected (original)", "max"
        ),
        total_adjusted_customer_count=pd.NamedAgg(
            "customers affected (adjusted)", "max"
        ),
        cost_to_trim_meta=pd.NamedAgg("cost to trim (BHT) [META + Sentinel-2]", "sum"),
        cost_to_trim_mjm=pd.NamedAgg("cost to trim (BHT) [MJM]", "sum"),
    ),
    left_index=True,
    right_index=True,
)

veg_units = veg_units.rename(
    columns={
        "risk": "risk (customer interruptions)",
        "average_probability_of_outage": "average probability of outage (%)",
        # "consequence_of_outage": "consequence of outage",
        "cost_to_trim_meta": "cost to trim (BHT) [META + Sentinel-2]",
        "cost_to_trim_mjm": "cost to trim (BHT) [MJM]",
        # "percent_vegetated": "Percent Vegetated [META + Sentinel-2]",
        "total_original_customer_count": "customers affected (original)",
        "total_adjusted_customer_count": "customers affected (adjusted)",
    }
)

veg_units["geometry"] = veg_units["geometry"].convex_hull
veg_units = veg_units.rename(columns={"geometry": "polygon"}).reset_index()
veg_units = gpd.GeoDataFrame(veg_units, geometry="polygon", crs=corridors_df.crs)
veg_units.head(3)

#### Add in the jurisdiction

In [ ]:
veg_units = veg_units.sjoin(
    aoj[["NAME", "geometry"]].rename(columns={"NAME": "Area of Jurisdiction"})
).drop("index_right", axis=1)
veg_units.head(3)

#### Organize veg units columns 

In [ ]:
veg_units = veg_units[
    [
        "Area of Jurisdiction",
        "feeder id",
        "polygon",
        "average probability of outage (%)",
        "cost to trim (BHT) [META + Sentinel-2]",
        "cost to trim (BHT) [MJM]",
        "customers affected (original)",
        "customers affected (adjusted)",
        "risk (customer interruptions)",
    ]
]

#### Save to file and DB

In [ ]:
io_utils.save_file_via_catalog(
    veg_units,
    "veg_clusters",
)

veg_units_sql = veg_units.copy()
veg_units_sql["polygon"] = veg_units_sql["polygon"].map(lambda x: x.wkt)
veg_units_sql = veg_units_sql.fillna(0)
veg_units_sql.map(lambda x: round(x, 2) if type(x) == float else x).to_sql(
    "veg_units", conn, if_exists="replace", index=False
)

#### Scenario Outcomes

No need to do any transformations. Just put it in the db.

In [ ]:
scenario_features = io_utils.load_file_from_catalog("optimization_metrics")
print(scenario_features.shape)
scenario_features.head()

In [ ]:
scenario_features.to_sql("veg_scenarios", conn, if_exists="replace", index=False)

#### Shap

No need to do any transformations. Just put it in the db.

In [ ]:
shap_features = io_utils.load_file_from_catalog("model_shap_features").rename(
    columns={"nearest_upstream_device": "nearest upstream device"}
)
print(shap_features.shape)
shap_features.head()

In [ ]:
shap_features.to_sql("veg_shap_features", conn, if_exists="replace", index=False)

#### Raw Asset Data

This is needed only for the purpose of making the shap charts

In [ ]:
values_df = io_utils.load_file_from_catalog("model_inputs_prediction").rename(
    columns={"nearest_upstream_device": "nearest upstream device"}
)
values_df.head(1)

In [ ]:
values_df[list(shap_features)].to_sql("veg_raw", conn, if_exists="replace", index=False)

#### Areas of Jursidction

In [ ]:
# UAT test notebook ran
print("Test 1 passed")